# See issue #52

In [ ]:
%matplotlib inline

In [ ]:
import json
from pathlib import Path
import os

from astropy.io import fits 
from astropy.wcs import WCS
import numpy as np
from tqdm import tqdm

from rgz.rgz import get_wcs
import rgz.constants
import rgz.subjects
from rgz import units as u

import matplotlib.pyplot as plt 
plt.ion()

In [ ]:
# Check whether RGZ subject contour coordinates are indexed from 0 or 1.
# WARNING: this takes about 20 minutes to run! 
cache_path = Path("data/cache")  # NOTE: the json files in here only contain FITS contours.
contour_coords = []
raw_subject_json_fnames = [f for f in os.listdir(cache_path) if f.endswith("json")]
for raw_subject_json_fname in tqdm(raw_subject_json_fnames):
    js = json.load(open(cache_path / raw_subject_json_fname))
    try:
        for ii in range(len(js["contours"])):
            for contour in js["contours"]:
                assert contour[0]["k"] == 0
                contour_coords += contour[0]["bbox"]
    except (KeyError, TypeError):
        continue
    
print(min(contour_coords))
print(max(contour_coords))

### Output:
```
100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 177284/177284 [14:42<00:00, 200.84it/s]1.0
201.00000000000003
```
So they appear to be indexed from 1.


In [ ]:
os.getcwd()

In [ ]:
data_path = Path("rgz/testdata")
cache_path = data_path / "first"

In [ ]:
# Load subjects
processed_subjects_fname = data_path / "subjects_processed.json"
subjects = rgz.subjects.read(processed_subjects_fname)
subjects = list(subjects)

In [ ]:
# Load a FIRST image and overlay bounding boxes on top to sanity-check. 
"""
The goal here is to check whether there is an off-by-one error in the code.
Currently it's assumed that the bbox coordinates given in the raw RGZ contour
data are indexed from 0, when they may actually be indexed from 1.
If there is such an error, then the radio blobs will NOT be aligned with the 
bboxes. 
"""
plt.close("all")
for idx in range(len(subjects)):
        subject = subjects[idx]
        id = subject.id
        fits_fname = cache_path / f"{id}.fits"

        # Get FIRST image 
        hdulist = fits.open(fits_fname)
        im = hdulist[0].data

        # Get zeroth contour level
        first_json = json.load(open(cache_path / f"{id}.json"))
        contour = [c for c in first_json["contours"][0] if c["k"] == 0][0]
        min_level = contour["level"]
        # TODO also plot contours over the top so it's more immediately obvious what's going on.

        # Overlay pixel coords read in directly from raw subject 
        for bb, bbox in enumerate(subject.bboxes.keys()):
                # Plot 
                fig, ax = plt.subplots(subplot_kw={"projection": subject.wcs})
                ax.imshow(im, vmax=min_level)
                
                # Get bbox coords
                xmin, ymin, xmax, ymax = bbox
                bbox_m1 = [b - 1 for b in bbox]
                # Make copies where it's assumed they are indexed from 1
                xmin_m1, ymin_m1, xmax_m1, ymax_m1 = bbox_m1
                

                # Flip up/down and scale. 
                # If coords are indexed from 1, then ranges should be [1, 132].
                # So flipping y = 132 should go to y = 1 so need to subtract 131, not 132.
                ymin = (rgz.constants.RADIO_MAX_PX - 1) - ymin
                ymax = (rgz.constants.RADIO_MAX_PX - 1) - ymax
                # If we've already subtracted 1, then the range should be [1, 131].
                # So flipping y = 131 should go to y = 0 so need to subtract 131 not 132.
                ymin_m1 = (rgz.constants.RADIO_MAX_PX - 1) - ymin_m1
                ymax_m1 = (rgz.constants.RADIO_MAX_PX - 1) - ymax_m1

                # Scale 
                xmin_m1 *= 100 / rgz.constants.RADIO_MAX_PX
                xmax_m1 *= 100 / rgz.constants.RADIO_MAX_PX
                ymin_m1 *= 100 / rgz.constants.RADIO_MAX_PX
                ymax_m1 *= 100 / rgz.constants.RADIO_MAX_PX
                xmin *= 100 / rgz.constants.RADIO_MAX_PX
                xmax *= 100 / rgz.constants.RADIO_MAX_PX
                ymin *= 100 / rgz.constants.RADIO_MAX_PX
                ymax *= 100 / rgz.constants.RADIO_MAX_PX

                ax.plot([xmin, xmin, xmax, xmax, xmin],
                        [ymin, ymax, ymax, ymin, ymin],
                        transform=ax.get_transform("pixel"),
                        color="r", lw=1, label="Original")

                ax.plot([xmin_m1, xmin_m1, xmax_m1, xmax_m1, xmin_m1],
                        [ymin_m1, ymax_m1, ymax_m1, ymin_m1, ymin_m1],
                        transform=ax.get_transform("pixel"),
                        color="b", lw=1, label="Indexed from zero")

                # Overlay RA/Dec coords transformed using the WCS with an origin of zero       
                xmin_deg, ymin_deg = subject.wcs.all_pix2world(np.array([[xmin, ymin]]), 0)[0]
                xmax_deg, ymax_deg = subject.wcs.all_pix2world(np.array([[xmax, ymax]]), 0)[0]
                ax.plot([xmin_deg, xmin_deg, xmax_deg, xmax_deg, xmin_deg],
                        [ymin_deg, ymax_deg, ymax_deg, ymin_deg, ymin_deg],
                        transform=ax.get_transform("fk5"),
                        color="r", ls="--", lw=3, label="Orignal - all_world2pix with origin = 0")
                
                # Repeat where it's assumed that bbox coordinates are indexed from 1
                xmin_m1_deg, ymin_m1_deg = subject.wcs.all_pix2world(np.array([[xmin_m1, ymin_m1]]), 0)[0]
                xmax_m1_deg, ymax_m1_deg = subject.wcs.all_pix2world(np.array([[xmax_m1, ymax_m1]]), 0)[0]
                ax.plot([xmin_m1_deg, xmin_m1_deg, xmax_m1_deg, xmax_m1_deg, xmin_m1_deg],
                        [ymin_m1_deg, ymax_m1_deg, ymax_m1_deg, ymin_m1_deg, ymin_m1_deg],
                        transform=ax.get_transform("fk5"),
                        color="b", ls="--", lw=3, label="Indexed from zero - all_world2pix with origin = 0")
                
                # Overlay RA/Dec coords transformed using the WCS with an origin of 1
                xmin_deg, ymin_deg = subject.wcs.all_pix2world(np.array([[xmin, ymin]]), 1)[0]
                xmax_deg, ymax_deg = subject.wcs.all_pix2world(np.array([[xmax, ymax]]), 1)[0]
                ax.plot([xmin_deg, xmin_deg, xmax_deg, xmax_deg, xmin_deg],
                        [ymin_deg, ymax_deg, ymax_deg, ymin_deg, ymin_deg],
                        transform=ax.get_transform("fk5"),
                        color="r", ls=":", lw=3, label="Orignal - all_world2pix with origin = 1")
                
                # Repeat where it's assumed that bbox coordinates are indexed from 1
                xmin_m1_deg, ymin_m1_deg = subject.wcs.all_pix2world(np.array([[xmin_m1, ymin_m1]]), 1)[0]
                xmax_m1_deg, ymax_m1_deg = subject.wcs.all_pix2world(np.array([[xmax_m1, ymax_m1]]), 1)[0]
                ax.plot([xmin_m1_deg, xmin_m1_deg, xmax_m1_deg, xmax_m1_deg, xmin_m1_deg],
                        [ymin_m1_deg, ymax_m1_deg, ymax_m1_deg, ymin_m1_deg, ymin_m1_deg],
                        transform=ax.get_transform("fk5"),
                        color="b", ls=":", lw=3, label="Indexed from zero - all_world2pix with origin = 1")
                
                # Repeat the above but using methods from rgz to check that they behave as 
                # expected
                phys_bbox = rgz.subjects.transform_bbox_px_to_phys(bbox_m1, subject.wcs)
                xmin_rgz_deg, ymin_rgz_deg, xmax_rgz_deg, ymax_rgz_deg = [p.value for p in phys_bbox]
                ax.plot([xmin_rgz_deg, xmin_rgz_deg, xmax_rgz_deg, xmax_rgz_deg, xmin_rgz_deg],
                        [ymin_rgz_deg, ymax_rgz_deg, ymax_rgz_deg, ymin_rgz_deg, ymin_rgz_deg],
                        transform=ax.get_transform("fk5"),
                        color="magenta", ls=":", lw=3, label="Original, transformed using rgz.subject.transform_bbox_px_to_phys")
                
                # Window in around radio blob
                xrng, yrng = xmax - xmin, ymax - ymin
                ax.set_xlim([xmin_m1 - xrng, xmax + xrng])
                ax.set_ylim([ymin_m1 - yrng, ymax + yrng])

                ax.legend(loc="center left", bbox_to_anchor=[1.05, 0.5])
                ax.set_title(f"{id}, bbox {bb}")

        